# rulesmith in ten minutes: rules for spotting fake job postings

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bradAGI/rulesmith/blob/main/notebooks/quickstart.ipynb)

This notebook gives rulesmith 100 labeled job postings and lets it search for a small
program of rules: cheap tests on a posting's fields, and a typed question to a language
model only where the fields cannot settle it. Then it scores what it found on postings it
never saw, beside the plain question asked of every posting.

It needs an OpenAI API key. `gpt-4.1` judges, since OpenAI's newer models do not return the
log-probabilities a judge is read from, and `gpt-6.1-sol` writes the rules. Three searches
make about a thousand short judge calls and a few dozen longer ones to write rules; that
should come to a few dollars, though we have not measured it. Nothing runs on a
GPU here, so a free Colab runtime is enough.

In [ ]:
# Installs rulesmith with the benchmark datasets, unless it is installed already.
try:
    import rulesmith  # noqa: F401
except ImportError:
    %pip install -q "rulesmith[bench] @ git+https://github.com/bradAGI/rulesmith"

## Settings

The judge answers the typed questions; the proposer writes the rules. Both default to
OpenAI. To use another OpenAI-compatible server, set `JUDGE_BASE_URL`, `PROPOSER_BASE_URL`
and the model names before running this cell, or edit them here.

In [ ]:
import getpass
import os
from pathlib import Path

JUDGE_URL = os.environ.get("JUDGE_BASE_URL", "https://api.openai.com/v1")
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "gpt-4.1")
PROPOSER_URL = os.environ.get("PROPOSER_BASE_URL", JUDGE_URL)
PROPOSER_MODEL = os.environ.get("PROPOSER_MODEL", "openai/gpt-6.1-sol")
# Extra request fields for the proposer, as JSON; a local Qwen3 needs its thinking turned off.
PROPOSER_OPTIONS = os.environ.get("PROPOSER_OPTIONS", "{}")
# Example scorings each search may spend; 200 is what the published benchmarks used.
BUDGET = int(os.environ.get("SEARCH_BUDGET", "200"))
# Searches to run; the one that scores best on validation is kept.
SEARCHES = int(os.environ.get("SEARCHES", "3"))
SEED = int(os.environ.get("SEED", "0"))

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

import subprocess

judge = ["--backend", "openai", "--model", JUDGE_MODEL, "--base-url", JUDGE_URL]


def rulesmith(*args, log=None):
    # Progress goes to a log, not the notebook; an error comes back with what rulesmith said.
    done = subprocess.run(["rulesmith", *map(str, args)], capture_output=True, text=True)
    if log:
        Path(log).write_text(done.stderr)
    if done.returncode:
        raise RuntimeError(f"rulesmith {args[0]} failed:\n{done.stderr[-3000:]}")
print("judge:", JUDGE_MODEL, "at", JUDGE_URL)
print("proposer:", PROPOSER_MODEL, "at", PROPOSER_URL)

## The task

Job postings from the [EMSCAD](https://huggingface.co/datasets/victor/real-or-fake-fake-jobposting-prediction)
dataset, sampled half fraudulent and half genuine: 50 to learn from, 50 to choose between
candidate programs, and 150 held out for the end. Each posting has text fields and a few
structured ones, such as whether it shows a company logo.

In [ ]:
import json
from collections import Counter

from rulesmith.bench import DATASETS, sample

task = sample(DATASETS["jobs"], 50, 50, 150, SEED)
Path("task.json").write_text(task.model_dump_json(indent=2, exclude_none=True))

print(task.description)
print("labels:", task.labels)
print("held out:", dict(Counter(example.label for example in task.test)))
print("fields of a posting:", sorted(task.train[0].state))

## The baseline: ask the model about every posting

One rule, which asks the judge the task's question over its labels.

In [ ]:
Path("direct.rules").write_text(f'always: ask "{task.description}" [task labels]\n')
rulesmith("evaluate", "task.json", "direct.rules", "--split", "test", *judge, "--output", "direct.json")

## The search

rulesmith starts from that one rule. It first mines rules a single field settles, then GEPA
asks the proposer for revisions and keeps the ones that score better on the 50 validation
postings. Each judge call costs a little of an example's score, so a rule that settles a
posting without one is worth something, and the starting question is kept unless a
winner's gain passes a sign test. These are the settings the benchmarks used.

One search can come back empty-handed: the proposer answers differently each time, and in
the benchmarks one search in three found no rule worth keeping. So this runs three, and
keeps the one that scored best on the validation postings; the held-out postings play no
part in the choice. Each search's log goes to `search-N.log`.

In [ ]:
scores = {}
for n in range(1, SEARCHES + 1):
    rulesmith(
        "optimize", "task.json", "direct.rules", *judge,
        "--reflection-model", PROPOSER_MODEL, "--reflection-base-url", PROPOSER_URL,
        "--reflection-options", PROPOSER_OPTIONS,
        "--mine", "--call-cost", "0.02", "--significance", "0.1",
        "--max-metric-calls", BUDGET, "--seed", SEED + n, "--output", f"search-{n}",
        log=f"search-{n}.log",
    )
    report = json.loads(Path(f"search-{n}/report.json").read_text())
    scores[n] = report["validation_score"]
    # A search's score is validation accuracy less 0.02 for every model call it made.
    print(f"search {n}: validation score {report['seed_validation_score']:.3f} where it started, "
          f"{report['validation_score']:.3f} for what it kept")

best = max(scores, key=scores.get)
print(f"\nkept search {best}:\n")
print(Path(f"search-{best}/plan.rules").read_text())

Those are the rules search kept. Each `read` names a field, each `when` is a rule tried
in order, and `ask` is the only line that calls the model.

## On postings it never saw

In [ ]:
rulesmith("evaluate", "task.json", f"search-{best}/plan.rules", "--split", "test", *judge, "--output", "searched.json")


def summary(path):
    scorecard = json.loads(Path(path).read_text())
    rows = scorecard["examples"]
    return scorecard["score"], sum(len(row["calls"]) for row in rows) / len(rows)


for name, path in [("ask about every posting", "direct.json"), ("searched rules", "searched.json")]:
    accuracy, calls = summary(path)
    print(f"{name:24}  accuracy {accuracy:.3f}   model calls per posting {calls:.2f}")

In our run all three searches beat where they started, and the rules they kept scored
0.753 against 0.733 for asking `gpt-4.1` about every posting, while asking it about one
posting in ten. Rules written by `gpt-4.1` instead scored 0.720 at 0.41 calls per posting,
and only one search in three improved: the model that writes the rules matters more than
the one that judges.

Rerun it and the numbers move: each search is one draw, and even the best of three can
find no rule worth keeping. The [benchmarks](https://github.com/bradAGI/rulesmith/blob/main/docs/benchmarks.md)
run three seeds per dataset against DSPy's classifiers and a TF-IDF baseline, and say where
rulesmith does not help.

## Next

- `rulesmith run search-N/plan.rules posting.json`, with N the search kept, decides one input, and names the rule that did.
- `rulesmith serve` answers decisions over HTTP; `rulesmith drift` checks production traces
  against what the rules were counted on.
- The [quickstart](https://github.com/bradAGI/rulesmith/blob/main/docs/quickstart.md) covers
  your own task file, and local judges (Ruling, Clef-flash) that need no API key.